In [ ]:
# =============================================================================
# 💻 데이터셋 제목: Defense Threat Detection Dataset (국방 위협 탐지 데이터셋)
# 🌐 데이터셋 의미: 이 데이터셋은 뉴스 기사나 시나리오 형태로 제시되는 텍스트에서
#              잠재적인 보안 위협(첩보, 사이버 전쟁, 경제 사보타주 등)을 식별하고
#              분류하는 능력을 AI 모델에게 학습시키기 위해 만들어졌습니다.
# 🚀 학습 목표: 자연어 처리(NLP)와 텍스트 분류를 연습하며, 특정 텍스트가 어떤 '위험'을 담고 있는지
#                   핵심 정보를 추출하는 방법을 배웁니다.
# =============================================================================

import random
import pprint
from datasets import load_dataset

# --- 설정 상수 ---
DATASET_NAME = "bayrameker/threat-detection"
SAMPLE_COUNT = 5  # 초보자 실습을 위해 상위 5개 샘플만 사용합니다!

# --- 데이터 로딩 함수 (Streaming 및 Fallback 처리) ---
def load_threat_dataset():
    """
    스트리밍 방식으로 데이터셋을 로드하고, 실패 시 일반 모드로 전환하는 함수.
    """
    print("✨ 데이터셋 로딩을 시도합니다...")
    
    # 1. 스트리밍 모드 우선 시도 (가장 빠르고 효율적입니다!)
    try:
        print("✅ 🚀 스트리밍 모드 (streaming=True)로 로드 시도 중...")
        # split='train'을 명시하여 로드합니다.
        dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
        print("✅ 스트리밍 로드 성공! 메모리 효율적으로 데이터를 처리할 수 있습니다.")
        return dataset
    
    except Exception as e:
        print(f"⚠️ 경고: 스트리밍 모드 로드에 실패했습니다. ({e})")
        print("🔄 일반 모드 (streaming=False)로 소량의 데이터를 다운로드하여 진행합니다.")
        
        # 2. 스트리밍 실패 시, 일반 Dataset 객체로 소량 로드
        try:
            # 에러 방지 및 진행을 위해 100개 정도만 다운로드합니다.
            dataset = load_dataset(DATASET_NAME, split='train')
            return dataset
        except Exception as e_fallback:
            print(f"❌ 심각한 오류: 데이터셋 로드에 최종 실패했습니다. ({e_fallback})")
            return None

# -----------------------------------------------------------------------------
# 🔑 데이터 로딩 및 샘플 확보
# -----------------------------------------------------------------------------

# 데이터셋 객체를 가져옵니다.
raw_dataset = load_threat_dataset()

if raw_dataset is None:
    print("데이터셋을 로드할 수 없어 실습을 종료합니다.")
    exit()

print(f"\n============================================================")
print(f"🧠 {DATASET_NAME} 데이터셋을 사용하여 {SAMPLE_COUNT}개의 샘플을 준비합니다.")

# 스트리밍 모드와 일반 모드에 따라 샘플링 방식을 다르게 적용합니다.
if hasattr(raw_dataset, "take"):
    # 스트리밍 데이터셋 (IterableDataset)인 경우: take() 사용
    print("➡️ [전략]: Streaming 방식으로 Take()하여 샘플을 준비합니다.")
    sampled_dataset_iterator = raw_dataset.take(SAMPLE_COUNT)
    # 리스트로 변환하여 반복 가능한 형태로 만듭니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 Dataset 객체인 경우: take() 사용
    print("➡️ [전략]: 일반 Dataset 방식으로 Take()하여 샘플을 준비합니다.")
    sampled_dataset = raw_dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sampled_dataset)

print(f"✅ 총 {len(sample_data_list)}개의 샘플을 확보했습니다. 이제부터 분석을 시작합니다!\n")


# =============================================================================
# 🚀 실습 1: 데이터 구조 탐색 (Threat Extraction)
# =============================================================================
print("============== [🏃‍♂️ 실습 1] 위협 정보 추출 및 구조 탐색 ==============")
print("👉 목표: 복잡하게 중첩된 'response' 리스트에서 핵심 위협 정보(Threat Type)만 정확히 꺼내봅시다!")

all_threat_types = []

# 샘플 데이터 리스트를 순회하며 분석을 진행합니다.
for i, sample in enumerate(sample_data_list):
    content = sample['content']
    responses = sample['response']
    
    # 하나의 샘플에서 발견된 모든 위협 타입을 리스트에 누적합니다.
    current_threats = [r['threat_type'] for r in responses]
    all_threat_types.extend(current_threats)
    
    print(f"\n--- [샘플 {i+1}/{SAMPLE_COUNT}] 분석 ---")
    print(f"📄 시나리오 (Content): {content[:60]}...")
    
    # 'response' 구조를 깔끔하게 출력하여 데이터 구조를 이해하도록 돕습니다.
    print(f"🚨 발견된 위협 정보 구조 (Response):")
    for response in responses:
        print(f"    - 🛡️ 타입: {response['threat_type']:<15} | 상세: {response['details']}")


# =============================================================================
# 💡 실습 2: 정량적 분석 (Statistical Insight)
# =============================================================================
print("\n\n============== [📊 실습 2] 데이터 기반 정량적 분석 ==============")
print("👉 목표: 이 데이터셋 전체에서 어떤 종류의 위협이 가장 많이 다뤄졌는지 통계를 내어봅시다!")

# 파이썬의 collections.Counter를 사용하지 않고, 기본적인 딕셔너리 카운팅을 사용해 원리를 이해합니다.
threat_counter = {}
for threat in all_threat_types:
    threat_counter[threat] = threat_counter.get(threat, 0) + 1

print("\n============================================================")
print("🔍 [분석 결과] 전체 샘플에서 발견된 위협 타입별 빈도수:")
print("============================================================")

# 위협 타입을 빈도수 순으로 정렬하여 보기 좋게 출력합니다.
sorted_threats = sorted(threat_counter.items(), key=lambda item: item[1], reverse=True)

for threat, count in sorted_threats:
    print(f"   - {threat: <15}: {count} 회 등장 (가장 많이 다뤄진 위협입니다!)")


# =============================================================================
# ✍️ 실습 3: LLM 프롬프트 생성기 (Creative Application)
# =============================================================================
print("\n\n============== [🤖 실습 3] 위협 인사이트 요약 프롬프트 생성기 ==============")
print("👉 목표: 사람이 이해하기 쉬운, LLM(거대 언어 모델)에 질문할 수 있는 멋진 프롬프트를 만들어봅시다!")

def generate_threat_insight_prompt(sample: dict) -> str:
    """
    하나의 샘플(시나리오)을 받아서, LLM이 분석할 수 있는 구조화된 프롬프트를 생성합니다.
    """
    content = sample['content']
    responses = sample['response']
    
    if not responses:
        return "🚨 경고: 이 시나리오는 명확한 위협 타입이 식별되지 않았습니다."

    # 모든 위협 타입과 상세 내용을 보기 좋게 리스트로 조합합니다.
    details_list = "\n".join(
        [f" - {r['threat_type']} (상세: {r['details']})" for r in responses]
    )

    # 최종 프롬프트 구조를 만듭니다.
    prompt = f"""
[TASK] 이 시나리오가 내포하고 있는 모든 위험과 잠재적 영향을 전문가의 관점에서 깊이 있게 분석해 주세요.

[CONTEXT]
---
시나리오 내용: "{content}"
---

[IDENTIFIED THREATS] (데이터셋에서 추출된 분석 결과):
{details_list}

[OUTPUT REQUIREMENT]
위 위협들을 바탕으로, 이 시나리오가 실제로 발생했을 때의 '최악의 결과' 3가지와, 이를 예방하기 위한 '구체적인 대응책' 3가지를 bullet point 형식으로 제시해 주세요.
"""
    return prompt.strip()

# 샘플 2번을 선택하여 프롬프트를 생성해봅니다.
if len(sample_data_list) >= 2:
    sample_to_analyze = sample_data_list[1]
    prompt_output = generate_threat_insight_prompt(sample_to_analyze)
    
    print("\n============================================================")
    print("📝 [결과] 2번째 샘플에 대한 LLM 분석 프롬프트:")
    print("============================================================")
    print(prompt_output)

print("\n🎉 축하합니다! 복잡한 AI 데이터셋을 성공적으로 로드하고, 분석하며, 실제 LLM에 사용할 프롬프트까지 생성했습니다!")
print("👏👏 코딩 실력을 한 단계 업그레이드하셨습니다! 계속 흥미로운 AI 프로젝트를 진행해 보세요!")